# Path B1 - GT crops + avaliação com e sem TTA+WBF

Este notebook treina o classificador B1 (ResNet-50) usando os crops GT pré-gerados em `processed_5cls/path_b/*/path_B/crops`.

Depois roda duas avaliações combinadas do sistema completo:

- detector YOLO + classificador, sem TTA+WBF;
- detector YOLO + classificador, com TTA multi-scale + WBF.

O treino não usa `--use_yolo_crops`, então o YOLO não gera crops durante o treinamento. O YOLO entra apenas na avaliação combinada.


In [ ]:
from pathlib import Path
import os
import sys
import shlex
import subprocess
import shutil
import torch
import pandas as pd
import json

WORKSPACE = Path('/workspace')
REPO_ROOT = WORKSPACE / 'TrashScan'

DATA_DIR = REPO_ROOT / 'data'
TRAIN_DIR = REPO_ROOT / 'train' / 'paths'
EVAL_DIR = REPO_ROOT / 'eval'

EXTERNAL_DIR = WORKSPACE / 'external_datasets'
TACO_DIR = WORKSPACE / 'TACO'
PROCESSED_DIR = WORKSPACE / 'processed_5cls' / 'path_b'

DATASET_YAML_PATH_B = PROCESSED_DIR / 'dataset_path_B.yaml'

RUNS_PATH_A_DIR = WORKSPACE / 'runs' / 'path_A'
RUNS_PATH_B_DIR = WORKSPACE / 'runs' / 'path_B_gt_b1'
MLFLOW_DIR = Path('/root/mlflow')
RESULTS_PATH_B_DIR = WORKSPACE / 'results_path_B'

TRAIN_PATH_B_SCRIPT = TRAIN_DIR / 'train_path_B.py'
EVAL_PATH_B_COMBINED_SCRIPT = EVAL_DIR / 'evaluate_path_B_combined.py'

for p in [RUNS_PATH_B_DIR, MLFLOW_DIR, RESULTS_PATH_B_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('Python             =', sys.executable)
print('REPO_ROOT          =', REPO_ROOT)
print('PROCESSED_DIR      =', PROCESSED_DIR)
print('DATASET_YAML_PATH_B=', DATASET_YAML_PATH_B)
print('RUNS_PATH_A_DIR    =', RUNS_PATH_A_DIR)
print('RUNS_PATH_B_DIR    =', RUNS_PATH_B_DIR)
print('RESULTS_PATH_B_DIR =', RESULTS_PATH_B_DIR)
print('TRAIN_PATH_B_SCRIPT=', TRAIN_PATH_B_SCRIPT)
print('EVAL_SCRIPT        =', EVAL_PATH_B_COMBINED_SCRIPT)


def run_cmd(cmd, cwd=WORKSPACE, env=None):
    print('$', ' '.join(shlex.quote(str(x)) for x in cmd))
    return subprocess.run([str(x) for x in cmd], cwd=str(cwd), env=env, check=True)


In [ ]:
if torch.cuda.is_available():
    DEVICE = '0'
    gpu_name = torch.cuda.get_device_properties(0).name
else:
    DEVICE = 'cpu'
    gpu_name = 'cpu'

print('Device:', DEVICE)
print('GPU:', gpu_name)

EPOCHS = 50
BATCH = 32
LR = 5e-5
PATIENCE = 10

CLASSIFIERS = [
    'resnet50',
]

EVAL_IMGSZ = 640
EVAL_DET_CONF = 0.001
EVAL_DET_IOU = 0.6

TTA_SCALES = ['512', '640', '768']
TTA_WBF_IOU = '0.55'
TTA_SKIP_BOX_THR = '0.001'


## 1) Selecionar detector YOLO

O treino em GT crops não usa o detector para gerar os exemplos. Ainda assim, o script pede `--detector_weights`, e a avaliação combinada precisa do detector.


In [ ]:
PREFERRED_DETECTOR = WORKSPACE / 'runs' / 'path_A_5cls' / 'yolov11m_o2o' / 'weights' / 'best.pt'
PATH_A_RUN_DIRS = [
    WORKSPACE / 'runs' / 'path_A_5cls',
    WORKSPACE / 'runs' / 'path_A',
    WORKSPACE / 'runs' / 'path_A_refined_head',
]


def read_yolo_results(run_dir: Path):
    best_pt = run_dir / 'weights' / 'best.pt'
    results_csv = run_dir / 'results.csv'
    metrics_json = run_dir / 'metrics.json'

    if not best_pt.exists():
        return None

    row = {
        'group': run_dir.parent.name,
        'model': run_dir.name,
        'run_dir': run_dir,
        'best_pt': best_pt,
        'mAP50_95': None,
        'mAP50': None,
        'precision': None,
        'recall': None,
        'source': None,
    }

    if results_csv.exists():
        df = pd.read_csv(results_csv)
        df.columns = [c.strip() for c in df.columns]
        map95_col = 'metrics/mAP50-95(B)'
        map50_col = 'metrics/mAP50(B)'
        precision_col = 'metrics/precision(B)'
        recall_col = 'metrics/recall(B)'

        if map95_col in df.columns:
            best_idx = df[map95_col].idxmax()
        elif map50_col in df.columns:
            best_idx = df[map50_col].idxmax()
        else:
            best_idx = df.index[-1]

        best = df.loc[best_idx]
        row['mAP50_95'] = float(best[map95_col]) if map95_col in df.columns else None
        row['mAP50'] = float(best[map50_col]) if map50_col in df.columns else None
        row['precision'] = float(best[precision_col]) if precision_col in df.columns else None
        row['recall'] = float(best[recall_col]) if recall_col in df.columns else None
        row['source'] = 'results.csv'
        return row

    if metrics_json.exists():
        with open(metrics_json, 'r') as f:
            m = json.load(f)
        row['mAP50_95'] = m.get('mAP50_95')
        row['mAP50'] = m.get('mAP50')
        row['precision'] = m.get('precision')
        row['recall'] = m.get('recall')
        row['source'] = 'metrics.json'
        return row

    row['source'] = 'weights_only'
    return row


if PREFERRED_DETECTOR.exists():
    DETECTOR_WEIGHTS = PREFERRED_DETECTOR
    print('Usando detector preferido:', DETECTOR_WEIGHTS)
else:
    records = []
    for base_dir in PATH_A_RUN_DIRS:
        if not base_dir.exists():
            print(f'[warn] Pasta não encontrada: {base_dir}')
            continue
        for run_dir in sorted(base_dir.iterdir()):
            if run_dir.is_dir():
                rec = read_yolo_results(run_dir)
                if rec is not None:
                    records.append(rec)

    df_detectors = pd.DataFrame(records)
    if df_detectors.empty:
        raise FileNotFoundError(
            'Nenhum detector com weights/best.pt foi encontrado em: '
            + ', '.join(str(p) for p in PATH_A_RUN_DIRS)
        )

    df_ranked = df_detectors.copy()
    df_ranked['rank_score'] = df_ranked['mAP50_95'].fillna(df_ranked['mAP50']).fillna(-1)
    df_ranked = df_ranked.sort_values(
        by=['rank_score', 'mAP50', 'precision', 'recall'],
        ascending=False,
        na_position='last',
    ).reset_index(drop=True)

    display(df_ranked[['group', 'model', 'mAP50_95', 'mAP50', 'precision', 'recall', 'source', 'best_pt']])
    DETECTOR_WEIGHTS = Path(df_ranked.iloc[0]['best_pt'])
    print('Melhor detector encontrado:', DETECTOR_WEIGHTS)

if not DETECTOR_WEIGHTS.exists():
    raise FileNotFoundError(f'Detector não encontrado: {DETECTOR_WEIGHTS}')


In [ ]:
# Preprocessamento dedicado do Path B. Rode uma vez; os outros notebooks GT reutilizam o mesmo root.
FORCE_PREPROCESS_PATH_B = False

PREPROCESS_PATH_B_SCRIPT = DATA_DIR / 'preprocess.py'
REQUIRED_PATH_B_ITEMS = [DATASET_YAML_PATH_B]

for split in ['train', 'val', 'test']:
    for subdir in ['images', 'labels', 'crops']:
        REQUIRED_PATH_B_ITEMS.append(PROCESSED_DIR / split / 'path_B' / subdir)

missing_path_b_items = [p for p in REQUIRED_PATH_B_ITEMS if not p.exists()]

if FORCE_PREPROCESS_PATH_B or missing_path_b_items:
    if FORCE_PREPROCESS_PATH_B:
        print('FORCE_PREPROCESS_PATH_B=True; executando preprocessamento do Path B.')
    else:
        print('Preprocessamento do Path B ausente ou incompleto. Itens faltantes:')
        for p in missing_path_b_items:
            print(' -', p)

    if not TACO_DIR.exists():
        raise FileNotFoundError(f'TACO_DIR não encontrado: {TACO_DIR}')
    if not PREPROCESS_PATH_B_SCRIPT.exists():
        raise FileNotFoundError(f'Script de preprocessamento não encontrado: {PREPROCESS_PATH_B_SCRIPT}')

    run_cmd([
        sys.executable, str(PREPROCESS_PATH_B_SCRIPT),
        '--taco_root', str(TACO_DIR),
        '--output_root', str(PROCESSED_DIR),
        '--path', 'B',
    ])
else:
    print('Preprocessamento do Path B já encontrado:', PROCESSED_DIR)


## 2) Conferir crops GT

Este notebook treina em `CropDataset`, então ele depende de `train/val/test/path_B/crops/{class_idx}`.


In [ ]:
for split in ['train', 'val', 'test']:
    crop_root = PROCESSED_DIR / split / 'path_B' / 'crops'
    if not crop_root.exists():
        raise FileNotFoundError(f'Crops GT não encontrados: {crop_root}')

    counts = {}
    for cls_dir in sorted(crop_root.iterdir()):
        if cls_dir.is_dir():
            counts[cls_dir.name] = len(list(cls_dir.glob('*.jpg')))

    print(split, crop_root)
    print('  total:', sum(counts.values()), 'por classe:', counts)


## 3) Treino Path B1 em GT crops

Sem `--use_yolo_crops`. Sem TTA+WBF no treino. O classificador é treinado nos crops GT pré-gerados.


In [ ]:
print('Parametros de treino:')
print('Detector weights:', DETECTOR_WEIGHTS)
print('Crops dir:', PROCESSED_DIR)
print('Output dir:', RUNS_PATH_B_DIR)
print('Classifiers:', CLASSIFIERS)
print('Epochs:', EPOCHS)
print('Batch:', BATCH)
print('Learning rate:', LR)
print('Patience:', PATIENCE)
print('Device:', DEVICE)
print('Use YOLO crops:', False)
print('TTA+WBF no treino:', False)


In [ ]:
run_cmd([
    sys.executable, str(TRAIN_PATH_B_SCRIPT),
    '--detector_weights', str(DETECTOR_WEIGHTS),
    '--crops_dir', str(PROCESSED_DIR),
    '--output', str(RUNS_PATH_B_DIR),
    '--classifiers', *CLASSIFIERS,
    '--epochs', str(EPOCHS),
    '--batch', str(BATCH),
    '--lr', str(LR),
    '--patience', str(PATIENCE),
    '--device', str(DEVICE),
])


## 4) Resumo do treino


In [ ]:
run_cmd([
    sys.executable, str(TRAIN_PATH_B_SCRIPT),
    '--detector_weights', str(DETECTOR_WEIGHTS),
    '--crops_dir', str(PROCESSED_DIR),
    '--output', str(RUNS_PATH_B_DIR),
    '--summarize',
])


## 5) Avaliação combinada sem TTA+WBF

Esta célula avalia detector + classificador no modo standard.


In [ ]:
CLASSIFIER_DIR = RUNS_PATH_B_DIR
DATA_YAML = DATASET_YAML_PATH_B
OUTPUT_DIR_STANDARD = RESULTS_PATH_B_DIR / 'b1_5cls_gt_standard'

cmd = [
    sys.executable, str(EVAL_PATH_B_COMBINED_SCRIPT),
    '--detector_weights', str(DETECTOR_WEIGHTS),
    '--classifier_dir', str(CLASSIFIER_DIR),
    '--classifiers', 'resnet50',
    '--data_yaml', str(DATA_YAML),
    '--output', str(OUTPUT_DIR_STANDARD),
    '--device', str(DEVICE),
    '--imgsz', str(EVAL_IMGSZ),
    '--det_conf', str(EVAL_DET_CONF),
    '--det_iou', str(EVAL_DET_IOU),
]

run_cmd(cmd)


## 6) Avaliação combinada com TTA+WBF

Esta célula avalia o mesmo classificador treinado em GT crops, mas com o detector usando TTA multi-scale + WBF.


In [ ]:
CLASSIFIER_DIR = RUNS_PATH_B_DIR
DATA_YAML = DATASET_YAML_PATH_B
OUTPUT_DIR_TTA_WBF = RESULTS_PATH_B_DIR / 'b1_5cls_gt_tta_wbf'

cmd = [
    sys.executable, str(EVAL_PATH_B_COMBINED_SCRIPT),
    '--detector_weights', str(DETECTOR_WEIGHTS),
    '--classifier_dir', str(CLASSIFIER_DIR),
    '--classifiers', 'resnet50',
    '--data_yaml', str(DATA_YAML),
    '--output', str(OUTPUT_DIR_TTA_WBF),
    '--device', str(DEVICE),
    '--imgsz', str(EVAL_IMGSZ),
    '--det_conf', str(EVAL_DET_CONF),
    '--det_iou', str(EVAL_DET_IOU),
    '--use_tta_wbf',
    '--tta_scales', *TTA_SCALES,
    '--tta_flip',
    '--tta_wbf_iou', TTA_WBF_IOU,
    '--tta_skip_box_thr', TTA_SKIP_BOX_THR,
]

run_cmd(cmd)


## 7) Comparar avaliações


In [ ]:
summary_paths = {
    'standard': OUTPUT_DIR_STANDARD / 'path_B_combined_summary.json',
    'tta_wbf': OUTPUT_DIR_TTA_WBF / 'path_B_combined_summary.json',
}

rows = []
for mode, path in summary_paths.items():
    if not path.exists():
        print(f'[warn] Resultado não encontrado para {mode}: {path}')
        continue
    data = json.loads(path.read_text())
    if isinstance(data, list):
        for row in data:
            row = dict(row)
            row['eval_mode'] = mode
            rows.append(row)
    else:
        row = dict(data)
        row['eval_mode'] = mode
        rows.append(row)

if rows:
    df = pd.DataFrame(rows)
    cols = [c for c in ['eval_mode', 'classifier', 'mAP50', 'mAP50_95', 'fps', 'inference_mode'] if c in df.columns]
    display(df[cols])
else:
    print('Nenhum resumo encontrado ainda.')
